# 02 — Feature Building
Build shots_df and team_windows_df, run validation checks, save to data/interim/.

In [1]:
import sys
sys.path.insert(0, '../src')

import pandas as pd

from config import RAW_DIR, MATCHES_FILE, INTERIM_DIR
from load_data import load_match_ids, load_competition_events
from features import build_team_windows, save_interim

In [2]:
# Load raw events
match_ids = load_match_ids(MATCHES_FILE)
events_df = load_competition_events(RAW_DIR, match_ids)
print(f'Loaded {len(events_df)} events across {events_df["match_id"].nunique()} matches')

Loaded 64 matches, 234652 events
  Shots: 1494  |  Goals: 195
  Unique event types (33): ['50/50', 'Bad Behaviour', 'Ball Receipt*', 'Ball Recovery', 'Block', 'Carry', 'Clearance', 'Dispossessed', 'Dribble', 'Dribbled Past', 'Duel', 'Error', 'Foul Committed', 'Foul Won', 'Goal Keeper', 'Half End', 'Half Start', 'Injury Stoppage', 'Interception', 'Miscontrol', 'Offside', 'Own Goal Against', 'Own Goal For', 'Pass', 'Player Off', 'Player On', 'Pressure', 'Referee Ball-Drop', 'Shield', 'Shot', 'Starting XI', 'Substitution', 'Tactical Shift']
Loaded 234652 events across 64 matches


In [3]:
# Extract shots DataFrame
shots_df = events_df[events_df['is_shot'] == 1].copy()
print(f'shots_df: {len(shots_df)} rows')
print(shots_df[['match_id','team_name','event_time_min','shot_xg','is_goal']].head())

shots_df: 1494 rows
     match_id team_name  event_time_min   shot_xg  is_goal
71    3857254   Tunisia        2.033333  0.023820        0
182   3857254   Tunisia        4.166667  0.014060        0
505   3857254   Tunisia       10.766667  0.033115        0
510   3857254   Tunisia       11.783333  0.043661        0
848   3857254   Denmark       22.083333  0.124033        0


In [4]:
# Build team windows (this takes a few minutes for 64 matches × 2 teams × 81 minutes)
print('Building team_windows_df...')
team_windows_df = build_team_windows(events_df)
print(f'team_windows_df: {len(team_windows_df)} rows')
print(f'Expected ~{64 * 2 * 81} rows (64 matches × 2 teams × 81 minutes)')
print(team_windows_df.head())

Building team_windows_df...
team_windows_df: 10368 rows
Expected ~10368 rows (64 matches × 2 teams × 81 minutes)
   match_id team_name opponent_name  t_minute  rolling_xg_5  shots_5  \
0   3857254   Denmark       Tunisia         5           0.0        0   
1   3857254   Denmark       Tunisia         6           0.0        0   
2   3857254   Denmark       Tunisia         7           0.0        0   
3   3857254   Denmark       Tunisia         8           0.0        0   
4   3857254   Denmark       Tunisia         9           0.0        0   

   opp_rolling_xg_5  opp_shots_5  xg_diff_5  shot_next_2  goal_next_5  \
0           0.03788            2   -0.03788            0            0   
1           0.03788            2   -0.03788            0            0   
2           0.03788            2   -0.03788            0            0   
3           0.01406            1   -0.01406            0            0   
4           0.01406            1   -0.01406            0            0   

  game_state  h

In [5]:
# Validation checks
print('=== Validation ===')

# No duplicates
dupes = team_windows_df.duplicated(subset=['match_id','team_name','t_minute']).sum()
print(f'Duplicate (match_id, team_name, t_minute): {dupes}  [expect 0]')

# xG ≥ 0
neg_xg = (team_windows_df['rolling_xg_5'] < 0).sum()
print(f'Negative rolling_xg_5: {neg_xg}  [expect 0]')

# Binary outcomes
print(f'shot_next_2 unique values: {sorted(team_windows_df["shot_next_2"].unique())}  [expect [0,1]]')
print(f'goal_next_5 unique values: {sorted(team_windows_df["goal_next_5"].unique())}  [expect [0,1]]')

# Game state distribution
print('\nGame state distribution:')
print(team_windows_df['game_state'].value_counts())

=== Validation ===
Duplicate (match_id, team_name, t_minute): 0  [expect 0]
Negative rolling_xg_5: 0  [expect 0]
shot_next_2 unique values: [0, 1]  [expect [0,1]]
goal_next_5 unique values: [0, 1]  [expect [0,1]]

Game state distribution:
game_state
tied        5502
trailing    2433
leading     2433
Name: count, dtype: int64


In [6]:
# Describe key features
print(team_windows_df[['rolling_xg_5','shots_5','xg_diff_5','shot_next_2','goal_next_5']].describe())

       rolling_xg_5       shots_5     xg_diff_5   shot_next_2   goal_next_5
count  10368.000000  10368.000000  1.036800e+04  10368.000000  10368.000000
mean       0.059987      0.556713  4.283268e-20      0.186053      0.065201
std        0.147105      0.825503  2.156955e-01      0.389168      0.246891
min        0.000000      0.000000 -1.170765e+00      0.000000      0.000000
25%        0.000000      0.000000 -3.597725e-02      0.000000      0.000000
50%        0.000000      0.000000  0.000000e+00      0.000000      0.000000
75%        0.050873      1.000000  3.597725e-02      0.000000      0.000000
max        1.170765      6.000000  1.170765e+00      1.000000      1.000000


In [7]:
# Save to interim
save_interim(shots_df, team_windows_df)
print('\nFiles saved to data/interim/')

Saved shots (1494 rows) → /Users/aidenpark/Project/cs109-soccer-momentum/data/interim/shots.csv
Saved team_windows (10368 rows) → /Users/aidenpark/Project/cs109-soccer-momentum/data/interim/team_windows.csv

Files saved to data/interim/
